# Exercise 2A One dimensional acoustic wave source-receiver exercise


## 1. problem setting

This notebook solves a one dimensional homogeneous acoustic wave problem with a localized source and a receiver benchmarked against a Green-function reference trace.

**Governing equation**

$$\frac{\partial^2 u}{\partial t^2}=c^2\frac{\partial^2u}{\partial x^2}+s(x,t),\qquad s(x,t)=f(t)\delta(x-x_s).$$

**Boundary and initial conditions**

$$u(0,t)=0,\qquad u(L,t)=0,\qquad u(x,0)=0,\qquad u_t(x,0)=0.$$

**Parameter table**

| Symbol | Meaning | Value |
|---|---:|---:|
| $L$ | domain length | 5000 m |
| $n_x$ | grid nodes | 501 |
| $c$ | wave speed | 334 m s$^{-1}$ |
| CFL | explicit stability factor | 0.45 |
| $f_0$ | source frequency parameter | 2.0 Hz |
| $x_s$ | source location | 2500 m |
| $x_r$ | receiver location | 3500 m |
| $t_{end}$ | configured final time | 6 s |


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from time import perf_counter
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.sparse import csr_matrix, diags, bmat, eye
from scipy.sparse.linalg import eigsh, factorized
from scipy.linalg import lu_factor, lu_solve
from IPython.display import HTML, display
from scipy.fft import dst, idst

Plot_COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000", "#7F7F7F", "#8B4513"]
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "axes.linewidth": 0.7,
    "axes.prop_cycle": plt.cycler(color=Plot_COLORS),
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "legend.fontsize": 6,
    "legend.frameon": False,
    "lines.linewidth": 1.1,
    "lines.markersize": 3,
    "image.cmap": "viridis",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "animation.embed_limit": 100,
})

CASE_ID = "Exercise2A"
ROOT = Path.cwd()
FIG = ROOT / "figures" / CASE_ID
OUT = ROOT / "outputs" / CASE_ID
FIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
                                                                    
    L: float = 5000.0               
    nx: int = 501
    c: float = 334.0                    
    cfl: float = 0.45                                   
    f0: float = 2.0
    t_end: float = 6
    nsave: int = 50
    source_x: float = 2500.0
    receiver_x: float = 3500.0

cfg = Config()
x = np.linspace(0, cfg.L, cfg.nx)
dx = x[1] - x[0]
isrc = int(np.argmin(np.abs(x - cfg.source_x)))
ir = int(np.argmin(np.abs(x - cfg.receiver_x)))
t0 = 2 / cfg.f0
t_end = min(cfg.t_end, cfg.L / cfg.c / 2) + t0

def source_time(t):
    """First derivative of a Gaussian, matching the course notebooks."""
    t = np.asarray(t, dtype=float)
    return -8.0 * (t - t0) * cfg.f0 * np.exp(-((4.0 * cfg.f0) ** 2) * (t - t0) ** 2)

def source_vector_1d(t):
    """Discrete point source s(x,t) ≈ src(t) δ(x-xs), scaled as in the course code."""
    s = np.zeros(cfg.nx)
    s[isrc] = source_time(t) / dx
    return s

def analytical_receiver_1d(times, dt_ref=None):
    """Receiver analytical seismogram from the 1D Green's function convolved with the source."""
    times = np.asarray(times, dtype=float)
    if len(times) == 0:
        return np.array([])
    if dt_ref is None:
        dt_ref = min(cfg.cfl * dx / cfg.c / 5.0, 5.0e-4)
    tmax = max(float(times.max()), cfg.t_end) + 8.0 * dt_ref
    tref = np.arange(0, tmax + dt_ref, dt_ref)
    r = abs(x[ir] - x[isrc])
    G = np.where(tref - r / cfg.c >= 0, 1.0 / (2.0 * cfg.c), 0)
    conv = np.convolve(G, source_time(tref) * dt_ref)[:len(tref)]
    return np.interp(times, tref, conv)

u0 = np.zeros_like(x)
v0 = np.zeros_like(x)

## 2. shared functions


In [ ]:
def Plot_axes(ax, grid=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.6, pad=2)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    if grid:
        ax.grid(True, color="0.88", linewidth=0.45, alpha=0.8)
    return ax

def linf(u, ref):
    return float(np.max(np.abs(np.asarray(u, dtype=float) - np.asarray(ref, dtype=float))))

def choose_snapshot_steps(nsteps, nsave):
    return set(np.unique(np.round(np.linspace(0, nsteps, min(nsave, nsteps + 1))).astype(int)))

def impose_bc(u):
    u = np.asarray(u, float).copy()
    u[0] = 0
    u[-1] = 0
    return u

def plot_snapshots_1d(x, snapshots, times, fname, title, value_label="p", time_scale=1.0, time_label=" s"):
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    idx = np.unique(np.round(np.linspace(0, len(times) - 1, min(10, len(times)))).astype(int))
    fig, axes = plt.subplots(2, 5, figsize=(7.2, 3.5), sharex=True, sharey=True)
    ymin = float(snapshots.min())
    ymax = float(snapshots.max())
    pad = max(0.06 * (ymax - ymin), 1e-8)
    panel_labels = list("abcdefghij")
    for k, ax in enumerate(axes.flat):
        if k < len(idx):
            j = idx[k]
            ax.plot(x, snapshots[j], color=Plot_COLORS[0], lw=1.15, label="numerical")
            ax.axvline(x[isrc], color="r", ls="--", lw=0.8, label="source" if k == 0 else None)
            ax.axvline(x[ir], color="k", ls=":", lw=0.8, label="receiver" if k == 0 else None)
            ax.set_ylim(ymin - pad, ymax + pad)
            ax.set_title(f"t={times[j] / time_scale:.3g}{time_label}", pad=2)
            Plot_axes(ax)
            ax.text(0.03, 0.94, f"({panel_labels[k]})", transform=ax.transAxes, ha="left", va="top", fontsize=7, fontweight="bold")
        else:
            ax.axis("off")
    for ax in axes[-1, :]: ax.set_xlabel("x [m]")
    for ax in axes[:, 0]: ax.set_ylabel(value_label)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.99, 1.03), ncol=3)
    fig.suptitle(title, y=1.02, fontsize=8)
    fig.tight_layout(w_pad=0.8, h_pad=0.9)
    fig.savefig(fname)
    plt.close(fig)
    return Path(fname)

def animate_snapshots_1d(x, snapshots, times, title, value_label="p", time_scale=1.0, time_label=" s", save_path=None):
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    ymin = float(snapshots.min())
    ymax = float(snapshots.max())
    pad = max(0.05 * (ymax - ymin), 1e-8)
    fig, ax = plt.subplots(figsize=(3.2, 3.0))
    line, = ax.plot([], [], lw=1.8, label="numerical")
    ax.axvline(x[isrc], color="r", ls="--", lw=0.8, label="source")
    ax.axvline(x[ir], color="k", ls=":", lw=0.8, label="receiver")
    ax.set_xlim(x[0], x[-1]); ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_xlabel("x [m]"); ax.set_ylabel(value_label); ax.set_title(title)
    Plot_axes(ax); ax.legend(loc="best")
    time_text = ax.text(0.03, 0.03, "", transform=ax.transAxes)
    def update(i):
        line.set_data(x, snapshots[i])
        time_text.set_text(f"t = {times[i] / time_scale:.3g}{time_label}")
        return line, time_text
    anim = FuncAnimation(fig, update, frames=len(times), interval=100, blit=True)
    if save_path is not None:
        try:
            anim.save(save_path, writer=PillowWriter(fps=10))
        except Exception as exc:
            print(f"Animation save skipped for {save_path}: {exc}")
    plt.close(fig)
    try:
        return HTML(anim.to_jshtml())
    except Exception:
        return None

def display_method_result_row(method_label, snapshot_path, anim_obj):
    print(method_label)
    print("snapshot figure:", snapshot_path)
    if anim_obj is not None:
        display(anim_obj)

def wave_error_summary(method, snapshots, times, elapsed, dt, nsteps):
    receiver_num = np.asarray(snapshots)[:, ir]
    receiver_exact = analytical_receiver_1d(times)
    abs_error = np.abs(receiver_num - receiver_exact)
    return {
        "method": method,
        "dt_s": dt,
        "steps": int(nsteps),
        "wall_time_s": elapsed,
        "time_per_step_s": elapsed / max(nsteps, 1),
        "final_receiver_abs_error": float(abs_error[-1]),
        "receiver_Linf_error": linf(receiver_num, receiver_exact),
        "receiver_abs_error_history": abs_error,
        "receiver_numerical": receiver_num,
        "receiver_analytical": receiver_exact,
    }

def save_wave_method(method_key, method_label, snapshots, times, elapsed, dt, nsteps, store):
    store[method_key] = wave_error_summary(method_label, snapshots, times, elapsed, dt, nsteps)
    store[method_key]["snapshots"] = snapshots
    store[method_key]["times"] = times
    snapshot_path = plot_snapshots_1d(x, snapshots, times, FIG / f"{method_key}_snapshots.png", f"{method_label}: snapshots")
    anim = animate_snapshots_1d(x, snapshots, times, f"{method_label}: animation", save_path=FIG / f"{method_key}_animation.gif")
    display_method_result_row(method_label, snapshot_path, anim)

results = {}

def sponge_sigma_1d(x_nodes=x, width_fraction=0.02, strength_factor=5.0):
    width = width_fraction * cfg.L
    sigma_max = strength_factor * cfg.c / width
    dist_to_boundary = np.minimum(x_nodes - x_nodes[0], x_nodes[-1] - x_nodes)
    r = np.clip((width - dist_to_boundary) / width, 0, 1.0)
    return sigma_max * r**2

sigma_sponge = sponge_sigma_1d()
sigma_sponge_int = sigma_sponge[1:-1]

dt_explicit_fdm_eval = cfg.cfl * dx / cfg.c
nsteps_explicit_fdm_eval = int(np.ceil(t_end / dt_explicit_fdm_eval))
dt_explicit_fdm_eval = t_end / nsteps_explicit_fdm_eval
                                                                                                                    
def fem_mk(n, h):
    # Assemble linear-element mass and stiffness matrices.
    # Keep boundary coupling terms for the interior solve.
    M = np.zeros((n, n)); K = np.zeros((n, n))
    Me = h / 6.0 * np.array([[2, 1], [1, 2]], float); Ke = 1.0 / h * np.array([[1, -1], [-1, 1]], float)
    for e in range(n - 1):
        sl = slice(e, e + 2); M[sl, sl] += Me; K[sl, sl] += Ke
    return csr_matrix(M), csr_matrix(K)

def fem_wave_stable_dt(Mii, Kii, safety=0.45):
    # Estimate the explicit wave time step from the maximum FEM eigenfrequency.
    """
    Stable explicit FEM wave timestep from the generalized eigenproblem
        K phi = lambda M phi.
    Sparse version avoids forming inv(M)K as a dense matrix.
    """
    lambda_max = eigsh(
        Kii,
        k=1,
        M=Mii,
        which="LM",
        return_eigenvectors=False,
    )[0]
    return min(safety * 2.0 / (cfg.c * np.sqrt(lambda_max)), t_end)

M_fem_base, K_fem_base = fem_mk(cfg.nx, dx)
interior = slice(1, -1)
Kii_fem = K_fem_base[interior, interior].tocsr()

                                                                               
u_bc = impose_bc(np.zeros_like(u0))
Kbc_fem = (K_fem_base[1:-1, 0].toarray().ravel() * u_bc[0]
           + K_fem_base[1:-1, -1].toarray().ravel() * u_bc[-1])

Mii_fem_consistent = M_fem_base[interior, interior].tocsr()
M_fem_lumped_diag = np.asarray(M_fem_base.sum(axis=1)).ravel()
M_fem_lumped_base = diags(M_fem_lumped_diag, 0, format="csr")
Mii_fem_lumped_diag = M_fem_lumped_diag[1:-1]
Mii_fem_lumped = diags(Mii_fem_lumped_diag, 0, format="csr")

Sigma_fem = diags(sigma_sponge_int, 0, format="csr")

dt_explicit_fem_consistent_eval = fem_wave_stable_dt(Mii_fem_consistent, Kii_fem)

dt_explicit_fem_lumped_eval = fem_wave_stable_dt(Mii_fem_lumped, Kii_fem)
                                                                                                                                             
dt_common = min(dt_explicit_fdm_eval, dt_explicit_fem_consistent_eval, dt_explicit_fem_lumped_eval)
nsteps_common = int(np.ceil(t_end / dt_common))
dt_common = t_end / nsteps_common
dt_explicit_fdm = dt_common
nsteps_explicit_fdm = nsteps_common
dt_explicit_fem_consistent = dt_common                                                                                    
nsteps_explicit_fem_consistent = nsteps_common                                                                 
dt_explicit_fem_lumped = dt_common                                                                                    
nsteps_explicit_fem_lumped = nsteps_common                                                                 
print(f"Common minimum dt = {dt_common:.6e} s, steps = {nsteps_common}")
print(f"FEM explicit dt eval, consistent mass = {dt_explicit_fem_consistent_eval:.6e} s; using common dt = {dt_explicit_fem_consistent:.6e} s, steps = {nsteps_explicit_fem_consistent}")                                                      
print(f"FEM explicit dt eval, lumped mass     = {dt_explicit_fem_lumped_eval:.6e} s; using common dt = {dt_explicit_fem_lumped:.6e} s, steps = {nsteps_explicit_fem_lumped}")                                                      

## 3. FDM explicit


In [ ]:
def fdm_explicit():
    # Build the finite-difference Laplacian on interior unknowns.
    # Advance with the explicit update using the CFL-limited time step.
    dt_fdm = dt_explicit_fdm
    nsteps_fdm = nsteps_explicit_fdm
    save_steps = choose_snapshot_steps(nsteps_fdm, cfg.nsave)

    n = len(u0)-2; D = diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1,0,1], format="csr")/dx**2
    u_bc = impose_bc(np.zeros_like(u0))                                                                                       
    bc = np.zeros(n); bc[0] += cfg.c**2 * u_bc[0] / dx**2; bc[-1] += cfg.c**2 * u_bc[-1] / dx**2
    u = impose_bc(u0)
    
    sigma_i = sigma_sponge_int
                                                  
    u_prev = u.copy()
    u_prev[1:-1] = u[1:-1] - dt_fdm * v0[1:-1] + 0.5 * dt_fdm**2 * (cfg.c**2 * (D @ u[1:-1]) + bc + source_vector_1d(0)[1:-1])
    u_prev = impose_bc(u_prev)
    snapshots = []; times = []; tic = perf_counter()
    for step in range(nsteps_fdm + 1):
        t = step * dt_fdm
        if step in save_steps: snapshots.append(u.copy()); times.append(t)
        if step == nsteps_fdm: break
        rhs = ((2.0 - sigma_i * dt_fdm) * u[1:-1]
               - (1.0 - sigma_i * dt_fdm) * u_prev[1:-1]
               + dt_fdm**2 * (cfg.c**2 * (D @ u[1:-1]) + bc + source_vector_1d(t)[1:-1]))
        un = u.copy()
        un[1:-1] = rhs
        un = impose_bc(un)

        u_prev, u = u, un

    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fdm, nsteps_fdm

snap, tt, elapsed, dt, ns = fdm_explicit()
print(f"FDM explicit Euler elapsed time: {elapsed:.6f} s")
save_wave_method("fdm_explicit", "FDM explicit Euler", snap, tt, elapsed, dt, ns, results)

## 4. FDM implicit


In [ ]:
def fdm_implicit():
    # Assemble the backward-Euler system matrix for one implicit time step.
    # Solve the linear system at each time level before applying boundary values.
    dt = dt_common                                                                      
    nsteps = nsteps_common
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)

    n=cfg.nx-2; D=diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1,0,1], format="csr")/dx**2
    u_bc = impose_bc(np.zeros_like(u0))                                                                                       
    bc = np.zeros(n); bc[0] += cfg.c**2 * u_bc[0] / dx**2; bc[-1] += cfg.c**2 * u_bc[-1] / dx**2

    I = eye(n, format="csr")
    Z = diags([np.zeros(n)], [0], format="csr")
    Sigma = diags([sigma_sponge_int], [0], format="csr")
    L = bmat([[Z, I], [cfg.c**2 * D, -Sigma]], format="csr")
    I_state = eye(2 * n, format="csr")
                                                
    A = I_state - dt * L
    solve_A = factorized(A.tocsc())
    y = np.r_[u0[1:-1], v0[1:-1]]
    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps + 1):
        t = step * dt
        if step in save_steps: u = np.zeros_like(x); u[1:-1] = y[:n]; snapshots.append(impose_bc(u)); times.append(t)
        if step == nsteps: break
        rhs = y + dt * np.r_[np.zeros(n), cfg.c**2 * bc + source_vector_1d(t + dt)[1:-1]]
        y = solve_A(rhs)
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = fdm_implicit()
print(f"FDM backward Euler elapsed time: {elapsed:.6f} s")
save_wave_method("fdm_backward_euler", "FDM backward Euler", snap, tt, elapsed, dt, ns, results)

## 5. FDM Crank--Nicolson


In [ ]:
def fdm_crank_nicolson():
    # Assemble Crank--Nicolson left and right time-stepping matrices.
    # Use midpoint diffusion/wave weighting for second-order time accuracy.
    dt = dt_common                                                                      
    nsteps = nsteps_common
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)

    n=cfg.nx-2; D=diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1,0,1], format="csr")/dx**2
    u_bc = impose_bc(np.zeros_like(u0))                                                                                       
    bc = np.zeros(n); bc[0] += cfg.c**2 * u_bc[0] / dx**2; bc[-1] += cfg.c**2 * u_bc[-1] / dx**2

    I = eye(n, format="csr")
    Z = diags([np.zeros(n)], [0], format="csr")
    Sigma = diags([sigma_sponge_int], [0], format="csr")
    L = bmat([[Z, I], [cfg.c**2 * D, -Sigma]], format="csr")
    I_state = eye(2 * n, format="csr")                                           

    A = I_state - 0.5 * dt * L
    B = I_state + 0.5 * dt * L
    solve_A = factorized(A.tocsc())
    y = np.r_[u0[1:-1], v0[1:-1]]
    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps + 1):
        t = step * dt
        if step in save_steps: u = np.zeros_like(x); u[1:-1] = y[:n]; snapshots.append(impose_bc(u)); times.append(t)
        if step == nsteps: break
        rhs = B @ y + 0.5 * dt * (np.r_[np.zeros(n), cfg.c**2 * bc + source_vector_1d(t)[1:-1]]
                                  + np.r_[np.zeros(n), cfg.c**2 * bc + source_vector_1d(t + dt)[1:-1]])
        y = solve_A(rhs)

    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = fdm_crank_nicolson()
print(f"FDM Crank--Nicolson elapsed time: {elapsed:.6f} s")
save_wave_method("fdm_crank_nicolson", "FDM Crank--Nicolson", snap, tt, elapsed, dt, ns, results)

## 6. Method 4 — FEM explicit, consistent and lumped mass


In [ ]:
def fem_explicit_consistent_and_lumped_mass(label, lumped=False):
    # Select either the consistent or lumped FEM mass matrix.
    # Use the FEM stability estimate to set the explicit time step.
    if lumped:
        Mii = Mii_fem_lumped
        Mbase = M_fem_lumped_base
        dt_fem = dt_explicit_fem_lumped
        nsteps_fem = nsteps_explicit_fem_lumped
    else:
        Mii = Mii_fem_consistent
        Mbase = M_fem_base
        dt_fem = dt_explicit_fem_consistent
        nsteps_fem = nsteps_explicit_fem_consistent

    def F(t):
        return np.asarray((Mbase @ source_vector_1d(t))[interior]).ravel()
    
    save_steps = choose_snapshot_steps(nsteps_fem, cfg.nsave)
    u = impose_bc(u0)
    solve_M = factorized(Mii.tocsc())
    rhs0 = F(0) - cfg.c**2 * (Kii_fem @ u[1:-1] + Kbc_fem) - Mii @ (Sigma_fem @ v0[1:-1])
    u_prev = u.copy()
    u_prev[1:-1] = u[1:-1] - dt_fem * v0[1:-1] + 0.5 * dt_fem**2 * solve_M(rhs0)
    u_prev = impose_bc(u_prev)
    snapshots = []
    times = []
    tic = perf_counter()
    for step in range(nsteps_fem  + 1):
        t = step * dt_fem
        if step in save_steps: snapshots.append(u.copy()); times.append(t)
        if step == nsteps_fem : break
        v_approx = (u[1:-1] - u_prev[1:-1]) / dt_fem
        rhs = F(t) - cfg.c**2 * (Kii_fem @ u[1:-1] + Kbc_fem) - Mii @ (Sigma_fem @ v_approx)
        un = u.copy()
        un[1:-1] = 2 * u[1:-1] - u_prev[1:-1] + dt_fem**2 * solve_M(rhs)
        un = impose_bc(un)
        u_prev, u = u, un
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fem, nsteps_fem 

for key, label, lumped in [
    ("fem_explicit_consistent", "FEM central difference, consistent mass", False),
    ("fem_explicit_lumped", "FEM central difference, lumped mass", True),
]:
    snap, tt, elapsed, dt, ns = fem_explicit_consistent_and_lumped_mass(label, lumped)
    print(f"{label} elapsed time: {elapsed:.6f} s")
    save_wave_method(key, label, snap, tt, elapsed, dt, ns, results)

## 7. Method 5 — FEM backward, consistent and lumped mass


In [ ]:
def fem_backward_consistent_and_lumped_mass(label, lumped=False):
    # Select either the consistent or lumped FEM mass matrix.
    # Assemble the backward FEM matrix or block system for implicit stepping.
    if lumped:
        Mii = Mii_fem_lumped
        Mbase = M_fem_lumped_base
        dt_ref_fem = dt_explicit_fem_lumped
    else:
        Mii = Mii_fem_consistent
        Mbase = M_fem_base
        dt_ref_fem = dt_explicit_fem_consistent

    dt_fem = dt_common                                                                                   
    nsteps_fem = nsteps_common
    n = cfg.nx - 2
    I = eye(n, format="csr")
    Z = csr_matrix((n, n))
    A = bmat(
        [
            [I, -dt_fem * I],
            [dt_fem * cfg.c**2 * Kii_fem, Mii + dt_fem * (Mii @ Sigma_fem)],
        ],
        format="csc",
    )
    solve_A = factorized(A)
    
    def F(t):
        return np.asarray((Mbase @ source_vector_1d(t))[interior]).ravel()
            
    y = np.r_[u0[1:-1], v0[1:-1]]
    save_steps = choose_snapshot_steps(nsteps_fem, cfg.nsave)
    snapshots = []
    times = []
    tic = perf_counter()
    for step in range(nsteps_fem + 1):
        t = step * dt_fem
        if step in save_steps: u = np.zeros_like(x); u[1:-1] = y[:n]; snapshots.append(impose_bc(u)); times.append(t)
        if step == nsteps_fem: break
        u_old = y[:n]
        v_old = y[n:]
        rhs_u = u_old
        rhs_v = Mii @ v_old + dt_fem * (F(t + dt_fem) - cfg.c**2 * Kbc_fem)
        rhs = np.r_[rhs_u, rhs_v]
        y = solve_A(rhs)
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fem, nsteps_fem

for key, label, lumped in [
    ("fem_implicit_consistent", "FEM backward Euler, consistent mass", False),
    ("fem_implicit_lumped", "FEM backward Euler, lumped mass", True),
]:
    snap, tt, elapsed, dt, ns = fem_backward_consistent_and_lumped_mass(label, lumped)
    print(f"{label} elapsed time: {elapsed:.6f} s")
    save_wave_method(key, label, snap, tt, elapsed, dt, ns, results)

## 8. Method 6 — pseudospectral sine method for heat or wave equation


In [ ]:
def pseudospectral_sine_method():
    # Transform the initial field and source into modal coefficients.
    # Advance spectral modes independently in time before reconstructing the field.
    """Pseudospectral modal solution using a sine basis.

    The sine basis corresponds to homogeneous Dirichlet boundary conditions,
    dq/dx = 0 at x=0 and x=L.  The mode m=0 is included.
    """
    m = np.arange(1, cfg.nx-1)
    k = m * np.pi / cfg.L
                       
    omega_m = cfg.c * k

    dt = dt_explicit_fdm
    nsteps = nsteps_explicit_fdm
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)
    
    def to_modal(u_int):
        return dst(u_int, type=1, norm="ortho")

    def to_phys(q):
        return idst(q, type=1, norm="ortho")
        
    u = impose_bc(u0)
    v = impose_bc(v0)

    q = to_modal(u[1:-1])
    q_t = to_modal(v[1:-1])                                                         
    b0 = to_modal(source_vector_1d(0)[1:-1])
                       
    q_prev = q - dt * q_t + 0.5 * dt**2 * (-(omega_m**2) * q + b0)
    snapshots = []
    times = []
    tic = perf_counter()
    for step in range(nsteps+1):
        t = step * dt
        if step in save_steps:
            u_out = np.zeros_like(x)
            u_out[1:-1] = to_phys(q)
            u_out = impose_bc(u_out)
            snapshots.append(u_out)
            times.append(t)
        if step == nsteps: break
        b = to_modal(source_vector_1d(t)[1:-1])
        q_dot = (q - q_prev) / dt
        u_dot = to_phys(q_dot)
        damping_modal = to_modal(sigma_sponge[1:-1] * u_dot)
        q_new = (2.0 * q - q_prev - dt**2 * (omega_m**2) * q - dt**2 * damping_modal + dt**2 * b)
        q_prev, q = q, q_new
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = pseudospectral_sine_method()
print(f"pseudospectral sine elapsed time: {elapsed:.6f} s")
save_wave_method("pseudospectral_sine_method", "Pseudospectral sine method", snap, tt, elapsed, dt, ns, results)

## 10. L2 error


In [ ]:
rows = []                                                                                
for key, v in results.items():
    times = np.asarray(v["times"], dtype=float)
                                                      
    u_num = np.asarray(v["snapshots"])[:, ir]                                                      
    u_ref = analytical_receiver_1d(times)
    diff = u_num - u_ref

    abs_err_t = np.abs(diff)
    rel_err_t = abs_err_t / np.maximum(np.abs(u_ref), 1e-300)

    abs_L2_t = np.sqrt(np.trapezoid(diff**2, x=times))
    ref_L2_t = np.sqrt(np.trapezoid(u_ref**2, x=times))
    rel_L2_t = abs_L2_t / max(ref_L2_t, 1e-300)                                                                            

    rows.append({
        "method": v["method"],
        "dt": v.get("dt_s", np.nan),                                                    
        "steps": v.get("steps", np.nan),
        "wall time": v.get("wall_time_s", np.nan),
        "temporal rel L2": float(rel_L2_t),
        "temporal abs L2": float(abs_L2_t),
        "final rel error": float(rel_err_t[-1]),
        "max rel error": float(np.max(rel_err_t)),
    })

green_l2_summary = pd.DataFrame(rows)[["method", "dt", "steps", "wall time", "temporal rel L2", "temporal abs L2", "final rel error", "max rel error"]].sort_values(
    "temporal rel L2"
)

display(green_l2_summary)

green_l2_summary.to_csv(
    OUT / "exercise2a_receiver_l2_error_vs_green_convolution.csv",
    index=False
)